## Games Embeddings

In [ ]:
# Add parent directory to sys.path to allow importing helpers and other modules
# This is required to use the embedding_service object, which is imported from helpers.embeddings_utils
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import psycopg2
from helpers.db_utils import get_football_connection_uri
from helpers.embeddings_utils import embedding_service

# connect to Azure postgres database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

# create product_desc table if it does not exist
cur.execute("""
    DROP TABLE IF EXISTS games_embeddings_diskann CASCADE;
    CREATE TABLE games_embeddings_diskann (
        vector_id SERIAL PRIMARY KEY,
        gameid INTEGER NOT NULL,
        gamedate DATE,
        gametimeeastern TIME,
        hometeamabbr VARCHAR,
        visitorteamabbr VARCHAR,
        week INTEGER,
        embedding_text VARCHAR,
        embedding_vector vector (1536) NOT NULL 
    );
""")
conn.commit()
cur.close()
cur = conn.cursor()

In [ ]:
#generate embeddings for product descriptions and store them in the product_desc table
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

cur.execute("SELECT * FROM games")
for row in cur.fetchall():
    gameId, gameDate, gametimeeastern, homeTeamAbbr, visitorTeamAbbr, week = row
    embedding_text = f"{gameId} {gameDate} {gametimeeastern} {homeTeamAbbr} {visitorTeamAbbr} {week}"
    response = await embedding_service.generate_embeddings([embedding_text])
    embedding_vector = response[0]
    cur.execute(
        "INSERT INTO games_embeddings_diskann (gameid, gamedate, gametimeeastern, hometeamabbr, visitorteamabbr, week, embedding_text, embedding_vector) VALUES (%s,%s,%s,%s,%s,%s,%s,%s)", 
        (gameId, gameDate,  gametimeeastern, homeTeamAbbr, visitorTeamAbbr, week, embedding_text, embedding_vector.tolist())
    )
print("All embeddings inserted into games_embeddings_diskann table.")
conn.commit()
cur.close()
conn.close()


In [ ]:
# Create index on the embedding vector column for efficient similarity search
query = """ CREATE INDEX games_embeddings_diskann_idx ON games_embeddings_diskann 
USING diskann (embedding_vector vector_cosine_ops)"""
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()
cur.execute(query)
conn.commit()
conn.close()

## Players Embeddings

In [ ]:
# Add parent directory to sys.path to allow importing helpers and other modules
# This is required to use the embedding_service object, which is imported from helpers.embeddings_utils
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import psycopg2
from helpers.db_utils import get_football_connection_uri
from helpers.embeddings_utils import embedding_service

# connect to Azure postgres database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

# create product_desc table if it does not exist
cur.execute("""
    DROP TABLE IF EXISTS players_embeddings_diskann CASCADE;
    CREATE TABLE players_embeddings_diskann (
        vector_id SERIAL PRIMARY KEY,
        nflid INTEGER NOT NULL,
        height VARCHAR,
        weight INTEGER,
        birthdate DATE,
        collegename VARCHAR,
        position VARCHAR,
        displayname VARCHAR,
        embedding_text VARCHAR,
        embedding_vector vector (1536) NOT NULL 
    );
""")
conn.commit()
cur.close()

In [ ]:
import psycopg2
from helpers.db_utils import get_football_connection_uri
from helpers.embeddings_utils import embedding_service

# connect to Azure postgres database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

#generate embeddings for product descriptions and store them in the product_desc table
cur.execute("SELECT * FROM players")
for row in cur.fetchall():
    nflId, height, weight, birthdate, collegename, position, displayname = row
    embedding_text = f"id:{nflId} ht:{height} wt:{weight} bday:{birthdate} coll/university:{collegename} pos:{position} name:{displayname}"
    response = await embedding_service.generate_embeddings([embedding_text])
    embedding_vector = response[0]
    cur.execute(
        "INSERT INTO players_embeddings_diskann (nflid, height, weight, birthdate, collegename, position, displayname, embedding_text, embedding_vector) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)", 
        (nflId, height, weight, birthdate, collegename, position, displayname, embedding_text, embedding_vector.tolist())
    )
print("All embeddings inserted into players_embeddings_diskann table.")
conn.commit()
cur.close()
conn.close()


In [ ]:
# Create index on the embedding vector column for efficient similarity search
query = """ CREATE INDEX players_embeddings_diskann_idx ON players_embeddings_diskann 
USING diskann (embedding_vector vector_cosine_ops)"""
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()
cur.execute(query)
conn.commit()
conn.close()

## Plays Embeddings

In [ ]:
# Add parent directory to sys.path to allow importing helpers and other modules
# This is required to use the embedding_service object, which is imported from helpers.embeddings_utils
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import psycopg2
from helpers.db_utils import get_football_connection_uri
from helpers.embeddings_utils import embedding_service

# connect to Azure postgres database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

# create product_desc table if it does not exist
cur.execute("""
    DROP TABLE IF EXISTS plays_embeddings_diskann CASCADE;
    CREATE TABLE plays_embeddings_diskann (
        vector_id SERIAL PRIMARY KEY,
        gameid VARCHAR,
        playid BIGINT,
        playdescription TEXT,
        quarter INTEGER,
        down INTEGER,
        yardstogo INTEGER,
        possessionteam VARCHAR,
        playtype VARCHAR,
        yardlineside VARCHAR,
        yardlinenumber BIGINT,
        offenseformation VARCHAR,
        personnelo VARCHAR,
        defendersinthebox BIGINT,
        numberofpassrushers BIGINT,
        personneld VARCHAR,
        typedropback VARCHAR,
        presnapvisitorScore BIGINT,
        presnaphomescore BIGINT,
        gameclock TIME,
        absoluteyardlinenumber BIGINT,
        penaltycodes VARCHAR,
        penaltyjerseynumbers VARCHAR,
        passresult VARCHAR,
        offenseplayresult BIGINT,
        playresult BIGINT,
        epa FLOAT,
        isDefensivepi BOOLEAN,
        embedding_text VARCHAR,
        embedding_vector vector (1536) NOT NULL 
    );
""")
conn.commit()
cur.close()

In [ ]:
import psycopg2
from helpers.db_utils import get_football_connection_uri
from helpers.embeddings_utils import embedding_service

# connect to Azure postgres database
conn_uri = get_football_connection_uri()
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()

#generate embeddings for product descriptions and store them in the product_desc table
cur.execute("SELECT * FROM plays")
for num, row in enumerate(cur.fetchall()):
    (
        gameid, playid, playdescription, quarter, down, yardstogo, possessionteam, playtype,
        yardlineside, yardlinenumber, offenseformation, personnelo, defendersinthebox,
        numberofpassrushers, personneld, typedropback, presnapvisitorScore, presnaphomescore,
        gameclock, absoluteyardlinenumber, penaltycodes, penaltyjerseynumbers, passresult,
        offenseplayresult, playresult, epa, isDefensivepi
    ) = row
    embedding_text = (
        f"gameid:{gameid} playid:{playid} playdescription:{playdescription} quarter:{quarter} down:{down} "
        f"yardstogo:{yardstogo} possessionteam:{possessionteam} playtype:{playtype} yardlineside:{yardlineside} "
        f"yardlinenumber:{yardlinenumber} offenseformation:{offenseformation} personnelo:{personnelo} "
        f"defendersinthebox:{defendersinthebox} numberofpassrushers:{numberofpassrushers} personneld:{personneld} "
        f"typedropback:{typedropback} presnapvisitorScore:{presnapvisitorScore} presnaphomescore:{presnaphomescore} "
        f"gameclock:{gameclock} absoluteyardlinenumber:{absoluteyardlinenumber} penaltycodes:{penaltycodes} "
        f"penaltyjerseynumbers:{penaltyjerseynumbers} passresult:{passresult} offenseplayresult:{offenseplayresult} "
        f"playresult:{playresult} epa:{epa} isDefensivepi:{isDefensivepi}"
    )
    response = await embedding_service.generate_embeddings([embedding_text])
    embedding_vector = response[0]
    cur.execute(
        "INSERT INTO plays_embeddings_diskann (gameid, playid, playdescription, quarter, down, yardstogo, possessionteam, playtype, yardlineside, yardlinenumber, offenseformation, personnelo, defendersinthebox, numberofpassrushers, personneld, typedropback, presnapvisitorScore, presnaphomescore, gameclock, absoluteyardlinenumber, penaltycodes, penaltyjerseynumbers, passresult, offenseplayresult, playresult, epa, isDefensivepi, embedding_text, embedding_vector) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", 
        (gameid, playid, playdescription, quarter, down, yardstogo, possessionteam, playtype, yardlineside, yardlinenumber, offenseformation, personnelo, defendersinthebox, numberofpassrushers, personneld, typedropback, presnapvisitorScore, presnaphomescore, gameclock, absoluteyardlinenumber, penaltycodes, penaltyjerseynumbers, passresult, offenseplayresult, playresult, epa, isDefensivepi, embedding_text, embedding_vector.tolist())
    )
    conn.commit()

print("All embeddings inserted into plays_embeddings_diskann table.")
conn.commit()
cur.close()
conn.close()


In [ ]:
# Create index on the embedding vector column for efficient similarity search
query = """ CREATE INDEX plays_embeddings_diskann_idx ON plays_embeddings_diskann 
USING diskann (embedding_vector vector_cosine_ops)"""
conn = psycopg2.connect(conn_uri)
cur = conn.cursor()
cur.execute(query)
conn.commit()
conn.close()